## Demo of client side Union of hotset dataset as view over Kafka through ISK and coldset dataset on MiniIO

In [1]:
# First shut down the packaged spark session

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

25/10/16 10:46:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
spark.stop()

In [6]:
spark = None

In [7]:
# Then reconnect with Spark Connect

In [8]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()


spark

In [9]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|   hotset|
|  coldset|
|   merged|
+---------+



In [10]:
spark.sql("use isk.hotset").show()
spark.sql("show tables").show()


++
||
++
++

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|   hotset|transactions_old|      false|
|   hotset|        accounts|      false|
|   hotset|       customers|      false|
|   hotset|        branches|      false|
|   hotset|    transactions|      false|
+---------+----------------+-----------+



In [11]:
# move some data into the coldset

In [12]:
spark.sql("DROP TABLE IF EXISTS direct.coldset.customers")

DataFrame[]

In [15]:
spark.sql("""
CREATE TABLE direct.coldset.customers 
USING iceberg 
TBLPROPERTIES('format-version'='2') 
PARTITIONED BY (kafka_partition, truncate(1000, kafka_offset)) 
AS 
  SELECT * 
  FROM isk.hotset.customers;
""")

DataFrame[]

In [16]:
spark.sql("SELECT COUNT(*) FROM direct.coldset.customers").show();

+--------+
|count(1)|
+--------+
|  400000|
+--------+



In [13]:
# From here we should do it with hyperstream

In [17]:
import requests
import json
from IPython.display import JSON

In [18]:
req = { 'set': 'COLD' }
x = requests.post('http://hyperstream:9088/api/schema',json = req)
JSON(x.text)

/usr/local/lib/python3.10/site-packages/IPython/core/display.py:664: UserWarning: JSON expects JSONable dict or list, not JSON strings
  warnings.warn("JSON expects JSONable dict or list, not JSON strings")


<IPython.core.display.JSON object>

In [19]:
req = { 
    'set': 'COLD',
    'index' : False,
    'sql' : 'SELECT * FROM customers LIMIT 10'
      }
x = requests.post('http://hyperstream:9088/api/query',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [21]:
req = { 
    'set': 'COLD',
    'index' : False,
    'sql' : 'SELECT * FROM customers WHERE Name=\'Brendan Yost\''
      }
x = requests.post('http://hyperstream:9088/api/query',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [22]:
req = { 
    'topic' : 'customers',
    'field' : 'Name'
      }
x = requests.put('http://hyperstream:9088/api/index',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [23]:
req = { 
    'topic' : 'customers',
    'field' : 'Name'
      }
x = requests.post('http://hyperstream:9088/api/index',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [28]:
req = { 
    'set': 'UNIFIED',
    'index' : True,
    'sql' : 'SELECT * FROM customers WHERE Name=\'Brendan Yost\''
      }
x = requests.post('http://hyperstream:9088/api/query',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [ ]:
# These are for investigating

In [25]:
spark.sql("DESCRIBE isk.merged.customers").show()

+---------------+--------------------+-------+
|       col_name|           data_type|comment|
+---------------+--------------------+-------+
|     CustomerID|              string|   NULL|
|           Name|              string|   NULL|
|        Address|              string|   NULL|
|          Email|              string|   NULL|
|        PhoneNo|              string|   NULL|
|kafka_partition|                 int|   NULL|
|   kafka_offset|              bigint|   NULL|
|       kafka_ts|       timestamp_ntz|   NULL|
|               |                    |       |
| # Partitioning|                    |       |
|         Part 0|     kafka_partition|       |
|         Part 1|truncate(1000, ka...|       |
+---------------+--------------------+-------+



In [35]:
spark.sql("USE cold.data").show();

++
||
++
++



In [30]:
spark.sql("SELECT *  FROM isk.merged.customers.files").show()

+-------+--------------------+-----------+-------+-----------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+--------------+---------------------+--------------------+
|content|           file_path|file_format|spec_id|  partition|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|referenced_data_file|content_offset|content_size_in_bytes|    readable_metrics|
+-------+--------------------+-----------+-------+-----------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+---------

In [27]:
spark.sql("SELECT * FROM (SELECT *  FROM isk.merged.customers WHERE (( kafka_partition = 0 AND kafka_offset >= 1000 AND kafka_offset <= 2000)))  WHERE Name = 'Brendan Yost'").show()

+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+
|CustomerID|        Name|             Address|               Email|       PhoneNo|kafka_partition|kafka_offset|            kafka_ts|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+
|    174178|Brendan Yost|Apt. 178 839 Bret...|su.wilderman@hotm...|(305) 225-5341|              0|        1000|2025-10-16 10:46:...|
+----------+------------+--------------------+--------------------+--------------+---------------+------------+--------------------+



In [27]:
spark.sql("SELECT * FROM cold.data.`index---customers---Name` LIMIT 10").show()

+-------------------+---------------+----------+----------+
|          index_key|kafka_partition|min_offset|max_offset|
+-------------------+---------------+----------+----------+
|     Steve Champlin|              0|     65000|     66000|
|        Gayle Kuhic|              0|     37000|     38000|
|         Shila Auer|              0|     37000|     38000|
|      Ivory Steuber|              0|     78000|     79000|
|    Angel Romaguera|              0|     24000|     25000|
|   Winford Lubowitz|              0|      8000|      9000|
|    Rozanne Kerluke|              0|     11000|     12000|
|        Adrien King|              0|     11000|     12000|
|        Chance Haag|              0|     48000|     49000|
|Wilbert Oberbrunner|              0|     62000|     63000|
+-------------------+---------------+----------+----------+



In [28]:
spark.sql("SELECT * FROM cold.data.streambased_metadata").show()

+---------+-----+---------------+----------+
|    topic|field|kafka_partition|max_offset|
+---------+-----+---------------+----------+
|customers| Name|              0|     79999|
+---------+-----+---------------+----------+



In [32]:
spark.sql("SELECT * FROM cold.data.`index---customers---Name` WHERE index_key = 'Brendan Yost' ORDER BY min_offset").show()

+------------+---------------+----------+----------+
|   index_key|kafka_partition|min_offset|max_offset|
+------------+---------------+----------+----------+
|Brendan Yost|              0|      1000|      2000|
+------------+---------------+----------+----------+



In [ ]:
import datetime

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers WHERE Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
WHERE c.Name = 'Brendan Yost' and c.kafka_offset >=0 and c.kafka_offset <=1000
""").show()
print( datetime.datetime.now())

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
JOIN cold.data.index_customers_name i
ON c.Name = i.Name AND c.kafka_partition = i.kafka_partition AND floor(c.kafka_offset/1000) = i.offset_batch
WHERE c.Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
JOIN (SELECT * FROM cold.data.index_customers_name WHERE Name='Brendan Yost') i
ON c.kafka_partition = i.kafka_partition AND floor(c.kafka_offset/1000) = i.offset_batch
WHERE c.Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())